# training-my-moe -- Interactive Code Tour

A hands-on, runnable companion to the `qwenmoe/` codebase: one notebook per
subpackage, each demoing every public class and function with real
executable code against this repo's own real artifacts (a real trained
checkpoint, real cached training data) -- no fabricated numbers, matching
this project's own documentation convention (see `journey_of_a_model/`).

For full mathematical derivations, papers, and worked-by-hand numeric
examples, see `journey_of_a_model/*.md` -- this tour intentionally stays
lighter: code + concise notes + live demos, one class/function at a time.

## Notebooks in this tour
| # | Notebook | Covers |
|---|----------|--------|
| 00 | this notebook | project map + a 60-second end-to-end tour |
| 01 | [01_config.ipynb](01_config.ipynb) | `qwenmoe/config.py`, `configs/presets.py` |
| 02 | [02_tokenizer.ipynb](02_tokenizer.ipynb) | `qwenmoe/tokenizer/` |
| 03 | [03_data.ipynb](03_data.ipynb) | `qwenmoe/data/` |
| 04 | [04_model.ipynb](04_model.ipynb) | `qwenmoe/model/` (norm, positional, attention, MoE, block, transformer, generate, trace) |
| 05 | [05_train.ipynb](05_train.ipynb) | `qwenmoe/train/` (trainer, scaling, compare) |
| 06 | [06_viz.ipynb](06_viz.ipynb) | `qwenmoe/viz/` (trace capture + Manim scenes) |
| 07 | [07_modernized_presets.ipynb](07_modernized_presets.ipynb) | `moe-modernization`: training `BASELINE_V2` / `FINEGRAINED` head-to-head |

This tour covers two specs: the original `qwen-moe-from-scratch` build (notebooks 01-06, the `SMOKE_TEST`/`BASELINE` architecture) and its `moe-modernization` follow-up (notebook 07, `BASELINE_V2`/`FINEGRAINED` -- QK-norm, sigmoid MoE routing, width-aware init, and DeepSeekMoE fine-grained experts). The end-to-end tour below touches both.

## How to use these notebooks
- Run cells top to bottom; select this project's `.venv` as the kernel.
- Most demos reuse real artifacts already on disk -- a real trained
  checkpoint under `runs/smoke_test/`, and real cached tiny-Shakespeare
  data under `.cache/data/` -- so almost everything below runs in well
  under a second. A couple of cells in notebooks 05/06 do a small amount
  of real training/rendering and take a few seconds.

In [1]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

import torch
import qwenmoe

print("torch version:", torch.__version__)
print("qwenmoe package location:", Path(qwenmoe.__file__).parent)

Working directory: /Users/sjoseph/Documents/Study/training-my-moe


torch version: 2.13.0
qwenmoe package location: /Users/sjoseph/Documents/Study/training-my-moe/qwenmoe


## Project map

```mermaid
flowchart LR
    CFG["config.py
    ModelConfig / TrainConfig / DataConfig"] --> TOK
    CFG --> DATA
    CFG --> MODEL
    TOK["tokenizer/
    CharTokenizer, ByteBPETokenizer"] --> DATA
    DATA["data/
    download, TokenDataset, fixed slices"] --> TRAIN
    MODEL["model/
    RMSNorm, positional, attention, MoE,
    TransformerBlock, QwenMoEModel, generate, trace"] --> TRAIN
    TRAIN["train/
    Trainer, scaling utilities, run_comparison"] --> VIZ
    MODEL --> VIZ["viz/
    capture_trace + Manim scenes"]
```

In [2]:
for path in sorted(Path("qwenmoe").rglob("*.py")):
    if path.name == "__init__.py":
        continue
    print(path)

qwenmoe/config.py
qwenmoe/data/dataset.py
qwenmoe/data/download.py
qwenmoe/model/attention.py
qwenmoe/model/block.py
qwenmoe/model/generate.py
qwenmoe/model/kv_cache.py
qwenmoe/model/moe.py
qwenmoe/model/mtp.py
qwenmoe/model/positional.py
qwenmoe/model/rmsnorm.py
qwenmoe/model/trace.py
qwenmoe/model/transformer.py
qwenmoe/tokenizer/base.py
qwenmoe/tokenizer/bpe.py
qwenmoe/tokenizer/char.py
qwenmoe/train/compare.py
qwenmoe/train/optim.py
qwenmoe/train/scaling.py
qwenmoe/train/trainer.py
qwenmoe/viz/common.py
qwenmoe/viz/scenes_attention.py
qwenmoe/viz/scenes_embedding.py
qwenmoe/viz/scenes_moe.py
qwenmoe/viz/scenes_norm_act.py
qwenmoe/viz/scenes_output.py
qwenmoe/viz/scenes_task10_preview.py
qwenmoe/viz/scenes_task11_preview.py
qwenmoe/viz/scenes_task15_preview.py
qwenmoe/viz/scenes_task3_preview.py
qwenmoe/viz/scenes_task5_preview.py
qwenmoe/viz/scenes_task6_preview.py
qwenmoe/viz/scenes_task7_preview.py
qwenmoe/viz/scenes_task8_preview.py
qwenmoe/viz/scenes_task9_preview.py
qwenmoe/vi

## A 60-second end-to-end tour
Before going module by module, let's touch every layer once: build a tiny
model from the real `SMOKE_TEST` preset, run one forward pass, then load
this repo's real trained checkpoint and generate text with it.

In [3]:
from configs.presets import SMOKE_TEST
from qwenmoe.model.transformer import QwenMoEModel
from qwenmoe.train.scaling import count_params

torch.manual_seed(0)
model = QwenMoEModel(SMOKE_TEST)
print(SMOKE_TEST)
print(count_params(model))

ModelConfig(vocab_size=65, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, d_ff=128, n_shared_experts=1, n_routed_experts=4, top_k=2, max_seq_len=128, positional_scheme='rope', activation='swiglu', moe_balancing='aux_loss', attention_variant='gqa', mla_latent_dim=None, lora_rank=None, tie_word_embeddings=True, rope_theta=10000.0, router_scoring='softmax', router_z_loss_weight=0.0, aux_loss_weight=0.01, seq_balance_loss_weight=0.0, qk_norm=False, mtp_depth=0, mtp_lambda=0.3, init_std=0.02)
ParamBreakdown(embedding=4224, attention_total=24704, moe_total=246400, moe_active=148096, total=275328, active=177024)


In [4]:
import torch.nn.functional as F

input_ids = torch.randint(0, SMOKE_TEST.vocab_size, (2, 16))
logits, extras = model(input_ids)
loss = F.cross_entropy(logits.reshape(-1, SMOKE_TEST.vocab_size), input_ids.reshape(-1))

print("logits shape:", logits.shape)
print("random-init loss:", loss.item())
print("ln(vocab_size)  :", torch.log(torch.tensor(float(SMOKE_TEST.vocab_size))).item())
print("(these two should be close -- a real bug in this project's task 10 was an")
print(" embedding init that blew this up to ~60 instead; see journey_of_a_model/010-training-loop.md)")

logits shape: torch.Size([2, 16, 65])
random-init loss: 3.1294376850128174
ln(vocab_size)  : 4.174387454986572
(these two should be close -- a real bug in this project's task 10 was an
 embedding init that blew this up to ~60 instead; see journey_of_a_model/010-training-loop.md)


In [5]:
from qwenmoe.model.generate import generate
from qwenmoe.tokenizer.base import build_tokenizer_from_meta
from qwenmoe.train.trainer import load_checkpoint

ckpt = load_checkpoint("runs/smoke_test/checkpoints/step_1000.pt")
trained_model = QwenMoEModel(ckpt["model_config"])
trained_model.load_state_dict(ckpt["model_state_dict"])
tokenizer = build_tokenizer_from_meta(ckpt["tokenizer_meta"])

torch.manual_seed(0)
print("TRAINED MODEL (step 1000):")
print(generate(trained_model, tokenizer, "ROMEO:", max_new_tokens=80, temperature=1.0, top_p=0.9))

TRAINED MODEL (step 1000):
ROMEO:
Thremh
Whit whonlt I doce meaman youd I tostet and fue yoXOBETI
Thee hath thigh


### Raw (untrained) vs. trained inference
`model` (built a few cells up) shares the exact same `SMOKE_TEST`
architecture and the same tokenizer, but was never trained -- its weights
are still the random initialization. Generating from it with the identical
decoding settings shows what "random weights" actually looks like as text,
in direct contrast to the trained checkpoint's output above.

In [6]:
torch.manual_seed(0)
print("RAW (untrained) MODEL:")
print(generate(model, tokenizer, "ROMEO:", max_new_tokens=80, temperature=1.0, top_p=0.9))

RAW (untrained) MODEL:


ROMEO:t3Zrj&hfkqkARxLs$ltUW?oocu  exhjyU&aNdkIgAlsvSBUMyccRD$wy$X- ntw
kVrM h&?j DJR!s



## The modernized model — `moe-modernization`

Everything above builds `BASELINE`, the architecture from the original
`qwen-moe-from-scratch` spec. A follow-up spec, `moe-modernization`
(`.kiro/specs/moe-modernization/`, 18 tasks, all complete), added ~20 new
`ModelConfig`/`TrainConfig` knobs — document-boundary attention masking,
Multi-Token Prediction, a KV cache, the Muon optimizer, a WSD schedule, and
more — each opt-in and defaulting to `BASELINE`'s exact pre-spec behavior.

Journal entry [034](../journey_of_a_model/034-modernized-presets.md) then
asked: of those ~20 knobs, which are safe enough (architecturally
non-destabilizing, ~free in parameters, backed by published practice) to
turn on **by default**? Three made the cut, giving `configs/presets.py::
BASELINE_V2`:

| change | from → to | why |
|---|---|---|
| `qk_norm` | `False → True` | per-head RMSNorm on q/k before RoPE — bounds the attention-logit scale regardless of depth (Qwen3; entry [022](../journey_of_a_model/022-qk-norm.md)) |
| `router_scoring` | `"softmax" → "sigmoid"` | independent per-expert affinities, renormalized over the selected `k` (DeepSeek-V3; entry [025](../journey_of_a_model/025-moe-routing-sigmoid-and-router-z-loss.md)) |
| `init_std` | `0.02 → √(2/(5·d_model))` | width-aware "small init" — `0.02` is GPT-2's `d_model=768` value, too small for this project's `d_model=256` (Nguyen & Salazar; entry [023](../journey_of_a_model/023-principled-initialization.md)) |

`configs/presets.py::FINEGRAINED` goes one step further: `BASELINE_V2` +
DeepSeekMoE fine-grained expert segmentation (8 experts of `d_ff=512`,
top-2 → 32 experts of `d_ff=128`, top-8 — the routed-FFN parameter budget
is held constant).

`BASELINE` and `SMOKE_TEST` are byte-for-byte untouched, so nothing above
this section changes.


In [7]:

import dataclasses

from configs.presets import BASELINE, BASELINE_V2, FINEGRAINED

# vocab_size=65 to match this section's real char tiny-Shakespeare checkpoints below.
for name, cfg in [("BASELINE", BASELINE), ("BASELINE_V2", BASELINE_V2), ("FINEGRAINED", FINEGRAINED)]:
    m = QwenMoEModel(dataclasses.replace(cfg, vocab_size=65))
    print(f"{name:12s} {count_params(m)}")

print("\nBASELINE_V2 vs BASELINE -- every field that differs:")
for f in dataclasses.fields(BASELINE):
    a, b = getattr(BASELINE, f.name), getattr(BASELINE_V2, f.name)
    if a != b:
        print(f"  {f.name:16s} {a!r} -> {b!r}")


BASELINE     ParamBreakdown(embedding=16896, attention_total=1181184, moe_total=21247488, moe_active=7091712, total=22445568, active=8289792)


BASELINE_V2  ParamBreakdown(embedding=16896, attention_total=1181568, moe_total=21247488, moe_active=7091712, total=22445952, active=8290176)


FINEGRAINED  ParamBreakdown(embedding=16896, attention_total=1181568, moe_total=19514880, moe_active=5359104, total=20713344, active=6557568)

BASELINE_V2 vs BASELINE -- every field that differs:
  router_scoring   'softmax' -> 'sigmoid'
  qk_norm          False -> True
  init_std         0.02 -> 0.0395



### Is `BASELINE_V2` actually better?

Journal [035](../journey_of_a_model/035-modernized-presets-training-run.md)
(and its driver, [07_modernized_presets.ipynb](07_modernized_presets.ipynb))
ran a real compute-matched, 2-seed `run_comparison` of all three presets on
an identical fixed token slice + budget. Measured result:

| variant | val loss (mean ± std) | vs `BASELINE` |
|---|---|---|
| `BASELINE` | 1.7836 ± 0.0193 | — |
| **`BASELINE_V2`** | **1.7114 ± 0.0150** | **−0.072 (3× the seed noise band — real)** |
| `FINEGRAINED` | 1.8506 ± 0.0098 | +0.067 (real, but *worse* — a scale/regime mismatch at this model size; see the entry) |

`BASELINE_V2` wins for +384 parameters. The cells below load the real
trained checkpoints from that comparison (`runs/modernized_presets/`) and
generate from both, the same "raw vs. trained" pattern as the `SMOKE_TEST`
tour above but comparing two *trained* architectures instead of trained vs.
untrained.


In [8]:

base_ckpt = load_checkpoint("runs/modernized_presets/BASELINE/seed_1337/checkpoints/step_305.pt")
v2_ckpt = load_checkpoint("runs/modernized_presets/BASELINE_V2/seed_1337/checkpoints/step_305.pt")

base_trained = QwenMoEModel(base_ckpt["model_config"])
base_trained.load_state_dict(base_ckpt["model_state_dict"])
base_trained.eval()

v2_trained = QwenMoEModel(v2_ckpt["model_config"])
v2_trained.load_state_dict(v2_ckpt["model_state_dict"])
v2_trained.eval()

char_tokenizer = build_tokenizer_from_meta(v2_ckpt["tokenizer_meta"])

torch.manual_seed(0)
print("BASELINE     (val loss 1.78 -- 305 steps, char tiny-Shakespeare):")
print(generate(base_trained, char_tokenizer, "ROMEO:", max_new_tokens=80, temperature=1.0, top_p=0.9))

print()
torch.manual_seed(0)
print("BASELINE_V2  (val loss 1.71 -- same steps/data/seed, qk_norm + sigmoid routing + wider init):")
print(generate(v2_trained, char_tokenizer, "ROMEO:", max_new_tokens=80, temperature=1.0, top_p=0.9))


BASELINE     (val loss 1.78 -- 305 steps, char tiny-Shakespeare):


ROMEO:
Thy more these not it off mean's you.
I to the recome with for beare hath think

BASELINE_V2  (val loss 1.71 -- same steps/data/seed, qk_norm + sigmoid routing + wider init):


ROMEO:
There me thereselted our in and by dange
The tranch upon his thee.

ROMEO:
Ay, 



Only 305 steps on a tiny char model, so neither output is fluent
Shakespeare — but it is the *same* comparison journal 035's numbers come
from, run live. See
[07_modernized_presets.ipynb](07_modernized_presets.ipynb) for the full
loss curves, expert-utilization, and real attention-logit traces that
explain *why* `BASELINE_V2` gets there faster (QK-norm keeps its attention
logits flat across all 6 layers, `[1.26, 1.33, 1.26, 1.27, 1.26, 1.29]`,
where `BASELINE`'s swing from `0.95` to `2.52`).


## Where to go next
Head to [01_config.ipynb](01_config.ipynb) to start at the bottom of the
stack, or jump straight to whichever module you're curious about using the
table above. To go deeper on the modernized model, see
[07_modernized_presets.ipynb](07_modernized_presets.ipynb) and journal
entries [034](../journey_of_a_model/034-modernized-presets.md) /
[035](../journey_of_a_model/035-modernized-presets-training-run.md).

For the full specs this project was built from, see
`.kiro/specs/qwen-moe-from-scratch/{requirements,design,tasks}.md` (the
original architecture) and `.kiro/specs/moe-modernization/{requirements,
design,tasks}.md` (the 2024-2026 modernization pass).